In [1]:
import numpy as np
import cv2

class BouncingBallSimulation:
    def __init__(self, high_res=(128, 128), low_res=(32, 32), ball_radius=4):
        self.high_res = high_res  # Rezolucija za preciznu simulaciju
        self.low_res = low_res    # Rezolucija za CNN (32x32)
        self.ball_radius = ball_radius
        self.obstacles = [] # Lista pravougaonika [x1, y1, x2, y2]

    def add_random_obstacles(self, n=3, min_size=15, max_size=30):
        """Dodaje nasumične pravougaone prepreke koje se ne preklapaju sa ivicama."""
        self.obstacles = []
        for _ in range(n):
            w = np.random.randint(min_size, max_size)
            h = np.random.randint(min_size, max_size)
            x = np.random.randint(self.ball_radius * 2, self.high_res[0] - w - self.ball_radius * 2)
            y = np.random.randint(self.ball_radius * 2, self.high_res[1] - h - self.ball_radius * 2)
            self.obstacles.append([x, y, x + w, y + h])

    def generate_episode(self, n_frames=20, velocity_range=(2, 5)):
        """Generiše jednu sekvencu kretanja loptice."""
        # Početna pozicija (pazimo da ne krene iz prepreke)
        pos = np.array([self.high_res[0] // 2, self.high_res[1] // 2], dtype=float)
        
        # Nasumična brzina
        angle = np.random.uniform(0, 2 * np.pi)
        speed = np.random.uniform(*velocity_range)
        vel = np.array([np.cos(angle) * speed, np.sin(angle) * speed])

        frames_high = []
        frames_low = []

        for t in range(n_frames):
            # 1. Iscrtavanje (High-res)
            frame = np.zeros(self.high_res, dtype=np.uint8)
            
            # Nacrtaj prepreke
            for obs in self.obstacles:
                cv2.rectangle(frame, (obs[0], obs[1]), (obs[2], obs[3]), 255, -1)
            
            # Nacrtaj lopticu (sa drugom vrednošću da bi VAE mogao da ih razlikuje ako želiš)
            # Ili koristi istu (255) za binarnu sliku
            cv2.circle(frame, (int(pos[0]), int(pos[1])), self.ball_radius, 255, -1)
            
            # 2. Downsample na 32x32 za CNN
            low_frame = cv2.resize(frame, self.low_res, interpolation=cv2.INTER_AREA)
            _, low_frame = cv2.threshold(low_frame, 127, 255, cv2.THRESH_BINARY)

            frames_high.append(frame)
            frames_low.append(low_frame)

            # 3. Fizika i kolizije (Euler integracija)
            pos += vel

            # Odbijanje od ivica kutije
            if pos[0] <= self.ball_radius or pos[0] >= self.high_res[0] - self.ball_radius:
                vel[0] *= -1
            if pos[1] <= self.ball_radius or pos[1] >= self.high_res[1] - self.ball_radius:
                vel[1] *= -1

            # Odbijanje od unutrašnjih prepreka
            for obs in self.obstacles:
                if (pos[0] + self.ball_radius > obs[0] and pos[0] - self.ball_radius < obs[2] and
                    pos[1] + self.ball_radius > obs[1] and pos[1] - self.ball_radius < obs[3]):
                    
                    # Odredi s koje strane je udarac (pojednostavljeno)
                    dx = min(abs(pos[0] - obs[0]), abs(pos[0] - obs[2]))
                    dy = min(abs(pos[1] - obs[1]), abs(pos[1] - obs[3]))
                    
                    if dx < dy:
                        vel[0] *= -1 # Udarac sa strane
                    else:
                        vel[1] *= -1 # Udarac odozgo/odozdo
                    break # Reši samo jednu koliziju po frejmu

        return np.array(frames_low), np.array(frames_high)

# Primer korišćenja:
sim = BouncingBallSimulation()
sim.add_random_obstacles(n=2)
dataset_low, dataset_high = sim.generate_episode(n_frames=30)

print(f"Shape za tvoj CNN: {dataset_low.shape}") # (30, 32, 32)

Shape za tvoj CNN: (30, 32, 32)


In [23]:
import numpy as np
import cv2
import imageio # pip install imageio

class BouncingBallSimulation:
    def __init__(self, high_res=(128, 128), low_res=(128, 128), ball_radius=2):
        self.high_res = high_res
        self.low_res = low_res
        self.ball_radius = ball_radius
        self.obstacles = []

    def add_random_obstacles(self, n=3, min_size=15, max_size=30):
        self.obstacles = []
        for _ in range(n):
            w = np.random.randint(min_size, max_size)
            h = np.random.randint(min_size, max_size)
            x = np.random.randint(self.ball_radius * 2, self.high_res[0] - w - self.ball_radius * 2)
            y = np.random.randint(self.ball_radius * 2, self.high_res[1] - h - self.ball_radius * 2)
            self.obstacles.append([x, y, x + w, y + h])

    def generate_episode(self, n_frames=20, velocity_range=(3, 5)):
        pos = np.array([self.high_res[0] // 2, self.high_res[1] // 2], dtype=float)
        angle = np.random.uniform(0, 2 * np.pi)
        speed = np.random.uniform(*velocity_range)
        vel = np.array([np.cos(angle) * speed, np.sin(angle) * speed])

        frames_low = []
        frames_high = []

        for t in range(n_frames):
            frame = np.zeros(self.high_res, dtype=np.uint8)
            
            # Iscrtaj prepreke (fiksni intenzitet 180)
            for obs in self.obstacles:
                cv2.rectangle(frame, (obs[0], obs[1]), (obs[2], obs[3]), 180, -1)
            
            # Iscrtaj lopticu (pun intenzitet 255)
            cv2.circle(frame, (int(pos[0]), int(pos[1])), self.ball_radius, 255, -1)
            
            # Resize za CNN
            low_frame = cv2.resize(frame, self.low_res, interpolation=cv2.INTER_AREA)
            
            frames_high.append(frame)
            frames_low.append(low_frame)

            # Fizika
            pos += vel
            if pos[0] <= self.ball_radius or pos[0] >= self.high_res[0] - self.ball_radius:
                vel[0] *= -1
            if pos[1] <= self.ball_radius or pos[1] >= self.high_res[1] - self.ball_radius:
                vel[1] *= -1

            for obs in self.obstacles:
                if (pos[0] + self.ball_radius > obs[0] and pos[0] - self.ball_radius < obs[2] and
                    pos[1] + self.ball_radius > obs[1] and pos[1] - self.ball_radius < obs[3]):
                    dx = min(abs(pos[0] - obs[0]), abs(pos[0] - obs[2]))
                    dy = min(abs(pos[1] - obs[1]), abs(pos[1] - obs[3]))
                    if dx < dy: vel[0] *= -1
                    else: vel[1] *= -1
                    break

        return np.array(frames_low), np.array(frames_high)

    def save_to_numpy(self, data, filename="dataset.npy"):
        """Saves as (N_Episodes, T, H, W)"""
        np.save(filename, data)
        print(f"Dataset sačuvan kao {filename}")

    def save_gif(self, frames, filename="simulation.gif", fps=10):
        """Generiše GIF od frejmova."""
        imageio.mimsave(filename, frames, fps=fps)
        print(f"GIF sačuvan kao {filename}")

# --- KORIŠĆENJE ---
sim = BouncingBallSimulation()
all_episodes = []

for i in range(5): # Generišemo 5 epizoda primera
    sim.add_random_obstacles(n=10)
    low_res_ep, high_res_ep = sim.generate_episode(n_frames=100)
    all_episodes.append(low_res_ep)
    
    # Sačuvaj prvu epizodu kao GIF da je vidiš
    if i == 0:
        sim.save_gif(high_res_ep, "provera_fizike.gif")

# Prebacujemo u oblik (N_episodes, T, H, W) pogodan za trening
final_data = np.stack(all_episodes)
sim.save_to_numpy(final_data)

GIF sačuvan kao provera_fizike.gif
Dataset sačuvan kao dataset.npy


In [56]:
import numpy as np 
np.random.default_rng(None).integers(2.4, 5)

np.int64(2)

In [119]:
import numpy as np
import cv2
import imageio

class BouncingBallSim:

    def __init__(self, size=(32,32), ball_radius=1, gravity=False, seed=None):
        self.H, self.W = size
        self.r = ball_radius
        self.gravity = gravity
        self.rng = np.random.default_rng(seed)
        self.obstacles = []

    def random_obstacles(self, n=2, min_size=4, max_size=10):
        """
        Obstacle generation
        """
        obs = []
        for _ in range(n):
            w = self.rng.integers(min_size, max_size)
            h = self.rng.integers(min_size, max_size)
            x = self.rng.integers(0, self.W - w)
            y = self.rng.integers(0, self.H - h)
            obs.append([x, y, x+w, y+h])
        self.obstacles = np.array(obs, dtype=np.int32)

    def step(self, pos, vel, substeps=4):
        """
        Update position and velocity using substeps for more stable collision physics.
        """
        dt = 1.0 / substeps

        for _ in range(substeps):
            # Apply gravity to actual velocity, proportional to dt
            if self.gravity:
                vel[1] += 0.2 * dt

            pos += vel * dt

            # Wall collisions
            if pos[0] < self.r or pos[0] > self.W - self.r:
                vel[0] *= -1
                pos[0] = np.clip(pos[0], self.r, self.W - self.r)
            if pos[1] < self.r or pos[1] > self.H - self.r:
                vel[1] *= -1
                pos[1] = np.clip(pos[1], self.r, self.H - self.r)

            # Obstacle collisions
            for x1, y1, x2, y2 in self.obstacles:
                if (pos[0] + self.r > x1 and pos[0] - self.r < x2 and pos[1] + self.r > y1 and pos[1] - self.r < y2):
                    cx = np.clip(pos[0], x1, x2)
                    cy = np.clip(pos[1], y1, y2)
                    dx, dy = pos[0]-cx, pos[1]-cy
                    
                    if dx*dx + dy*dy < self.r*self.r:
                        if abs(dx) > abs(dy):
                            vel[0] *= -1
                        else:
                            vel[1] *= -1

        return pos, vel

    def render(self, pos):
        """
        Renders the current frame with obstacles and the moving object.
        Uses uint8 for memory efficiency (0-255).
        """
        img = np.zeros((self.H, self.W), dtype=np.uint8)

        for r in self.obstacles:
            # Draw obstacles with anti-aliasing
            cv2.rectangle(img, (r[0], r[1]), (r[2], r[3]), 180, -1, lineType=cv2.LINE_AA)

        # Draw ball
        cv2.circle(img, (int(pos[0]), int(pos[1])), int(self.r), 255, -1, lineType=cv2.LINE_AA)

        return img

    def generate_episode(self, T=50, speed=(3,5)):
        """
        Generates a sequence of the simulation.
        """
        max_attempts = 20
        for _ in range(max_attempts):
            pos = np.array([self.rng.uniform(self.r, self.W - self.r),
                            self.rng.uniform(self.r, self.H - self.r)])
            if not any((r[0]-2*self.r < pos[0] < r[2]+2*self.r and
                        r[1]-2*self.r < pos[1] < r[3]+2*self.r) for r in self.obstacles):
                break
        else:
            pos = np.array([self.W/2, self.H/2])
            self.random_obstacles(n=0)

        angle = self.rng.uniform(0, 2*np.pi)
        v = self.rng.uniform(*speed)
        vel = np.array([np.cos(angle)*v, np.sin(angle)*v])

        frames = []
        for _ in range(T):
            frames.append(self.render(pos))
            pos, vel = self.step(pos, vel, substeps=4)

        return np.stack(frames)

# Example usage
sim = BouncingBallSim(size=(64,64), ball_radius=2, gravity=False)
sim.random_obstacles(n=2)
frames = sim.generate_episode(T=60)

imageio.mimsave("ball_lowres_gravity.gif", frames, fps=10)

In [83]:
xd = low[0, ...]

In [65]:
dataset = generate_dataset(sim,episodes=100000,T=20)

#np.save("balls_dataset.npy",dataset)

class BallDataset(Dataset):

    def __init__(self,path):

        self.data = np.load(path)

    def __len__(self):

        return len(self.data)

    def __getitem__(self,idx):

        x = self.data[idx,:-1]
        y = self.data[idx,1:]

        x = torch.tensor(x).unsqueeze(1).float()
        y = torch.tensor(y).unsqueeze(1).float()

        return x,y
    
from torch.utils.data import DataLoader

dataset = BallDataset("balls_dataset.npy")

loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True
)

episode 0


KeyboardInterrupt: 

In [ ]:
import numpy as np
import cv2
import torch
from torch.utils.data import Dataset
import imageio

class BouncingBallSim:

    def __init__(self, high_res=(128,128), low_res=(32,32), ball_radius=3, seed=None):
        self.H, self.W = high_res
        self.low_res = low_res
        self.r = ball_radius

        self.rng = np.random.default_rng(seed)
        self.obstacles = []

    def random_obstacles(self, n=6, min_size=0.1, max_size=0.25):
        """
        Obstacle generation
        """
        obs = []

        for _ in range(n):
            w = self.rng.integers(self.W*min_size, self.W*max_size)
            h = self.rng.integers(self.H*min_size, self.H*max_size)

            x = self.rng.integers(0, self.W-w)
            y = self.rng.integers(0, self.H-h)

            obs.append([x, y, x+w, y+h])

        self.obstacles = np.array(obs, dtype=np.int32)

    def step(self, pos, vel, substeps=4):
        """
        Updates position and velocity using sub-stepping for collision physics.
        """
        step_vel = vel / substeps

        for _ in range(substeps):
            pos += step_vel

            # Walls
            if pos[0] < self.r or pos[0] > self.W - self.r:
                step_vel[0] *= -1
                pos[0] = np.clip(pos[0], self.r, self.W - self.r)

            if pos[1] < self.r or pos[1] > self.H - self.r:
                step_vel[1] *= -1
                pos[1] = np.clip(pos[1], self.r, self.H - self.r)

            # Obstacles - brža provera preko bounding box pre filtera
            for x1, y1, x2, y2 in self.obstacles:
                if (pos[0] + self.r > x1 and pos[0] - self.r < x2 and
                    pos[1] + self.r > y1 and pos[1] - self.r < y2):
                    # kolizija sa boxom
                    cx = np.clip(pos[0], x1, x2)
                    cy = np.clip(pos[1], y1, y2)
                    dx, dy = pos[0]-cx, pos[1]-cy
                    if dx*dx + dy*dy < self.r*self.r:
                        if abs(dx) > abs(dy):
                            step_vel[0] *= -1
                        else:
                            step_vel[1] *= -1

        vel[:] = step_vel * substeps
        return pos, vel

    def render(self, pos):
        """
        Renders the current frame with obstacles and the moving object.
        Uses uint8 for memory efficiency (0-255).
        """
        img = np.zeros((self.H, self.W), dtype=np.uint8)

        for r in self.obstacles:
            # Draw obstacles with anti-aliasing
            cv2.rectangle(img, (r[0], r[1]), (r[2], r[3]), 180, -1, lineType=cv2.LINE_AA)

        # Draw ball
        cv2.circle(img, (int(pos[0]), int(pos[1])), int(self.r), 255, -1, lineType=cv2.LINE_AA)

        return img

    def generate_episode(self, T=50, speed=(3,5)):
        """
        Generates a sequence of high and low resolution frames of the simulation.
        Includes safeguard for starting position so it doesn't start inside obstacles.
        """
        max_attempts = 100
        for _ in range(max_attempts):
            pos = np.array([self.rng.uniform(0.15*self.W, 0.85*self.W),
                            self.rng.uniform(0.15*self.H, 0.85*self.H)], dtype=np.float32)
            if not any((r[0]-2*self.r < pos[0] < r[2]+2*self.r and
                        r[1]-2*self.r < pos[1] < r[3]+2*self.r) for r in self.obstacles):
                break
        else:
            # fallback ako ne nađe poziciju posle max_attempts
            pos = np.array([self.W/2, self.H/2], dtype=np.float32)

        # Random direction and speed
        angle = self.rng.uniform(0, 2*np.pi)
        v = self.rng.uniform(*speed)
        vel = np.array([np.cos(angle)*v, np.sin(angle)*v], dtype=np.float32)

        high, low = [], []

        for _ in range(T):
            frame = self.render(pos)
            high.append(frame)
            # Downsample frame for low-resolution output
            low.append(cv2.resize(frame, self.low_res, interpolation=cv2.INTER_AREA))
            pos, vel = self.step(pos, vel)

        return np.stack(low), np.stack(high)

def generate_dataset(sim, episodes=10000, T=30):
    """
    Creates a large-scale dataset of physics simulations.
    Uses uint8 low-res frames for memory efficiency.
    """
    # Pre-allocate memory for the entire dataset
    data = np.zeros((episodes, T, sim.low_res[0], sim.low_res[1]), dtype=np.uint8)

    for i in range(episodes):
        # Randomize environment layout for each episode
        sim.random_obstacles()

        # Generate low-res frames and store in the array
        low, _ = sim.generate_episode(T)
        data[i] = low

        # Progress tracking
        if i % 1000 == 0:
            print("episode", i)

    return data

# Primer korišćenja
sim = BouncingBallSim()
sim.random_obstacles()

low, high = sim.generate_episode(T=100)

# Sačuvaj animaciju u GIF
imageio.mimsave("ball.gif", high, fps=15)